In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        factor = 2 if bilinear else 1
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024 // factor))
        
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from tqdm import tqdm
# Check for GPU
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Paths - Update these to match your specific Kaggle input directories
# Updated Paths
IMAGE_DIR = "/kaggle/input/datasets/navaneethnemapu/palm-leaf-segmentation/dataset_12K/out_original_aug"
MASK_DIR = "/kaggle/input/datasets/navaneethnemapu/palm-leaf-segmentation/dataset_12K/out_groundtruth_aug"
MODEL_SAVE_PATH = "/kaggle/working/best_model.pth"

# Training Parameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 50

In [ ]:
import os

print(f"Current IMAGE_DIR path: {IMAGE_DIR}")

# Check if the path actually exists
if not os.path.exists(IMAGE_DIR):
    print("❌ ERROR: This folder does not exist at all.")
else:
    # List whatever is inside
    contents = os.listdir(IMAGE_DIR)
    print(f"Found {len(contents)} items in this folder.")
    print(f"First 10 items: {contents[:10]}")

In [ ]:
class PalmLeafPatchDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        # Only list valid image files
# Convert the filename to lowercase before checking the extension
        self.images = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))]
    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        # Assumes masks have the exact same filename as the images
        mask_path = os.path.join(self.mask_dir, self.images[idx]) 
        
        # Load image (RGB) and mask (Grayscale)
        image = np.array(Image.open(img_path).convert("RGB"), dtype=np.float32)
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)

        # Normalize to [0, 1]
        image = image / 255.0
        mask = mask / 255.0
        
        # Binarize the mask to ensure clear foreground/background
        mask[mask >= 0.5] = 1.0
        mask[mask < 0.5] = 0.0

        # PyTorch expects channels first: (C, H, W)
        image = np.transpose(image, (2, 0, 1))
        # Mask needs a channel dimension: (1, H, W)
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(image), torch.tensor(mask)

In [ ]:
# 1. Load Data
dataset = PalmLeafPatchDataset(image_dir=IMAGE_DIR, mask_dir=MASK_DIR)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
print(f"Total training patches loaded: {len(dataset)}")

# 2. Initialize Model
# Note: UNet(3, 1) means 3 input channels (RGB), 1 output channel (Binary Mask)
model = UNet(3, 1).to(DEVICE) 

# 3. Setup Loss and Optimizer
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

In [ ]:
def calculate_metrics(preds, targets, threshold=0.5, eps=1e-7):
    """
    Calculates the Mean Intersection over Union (mIoU) and Dice Coefficient.
    """
    # 1. Apply sigmoid to convert raw logits to probabilities
    probs = torch.sigmoid(preds)
    
    # 2. Convert probabilities to binary predictions (0 or 1)
    preds_bin = (probs > threshold).float()
    
    # 3. Flatten the tensors to easily compute intersection and union
    preds_flat = preds_bin.view(-1)
    targets_flat = targets.view(-1)
    
    # 4. Calculate core components
    intersection = (preds_flat * targets_flat).sum()
    total = preds_flat.sum() + targets_flat.sum()
    union = total - intersection
    
    # 5. Compute metrics (adding eps to avoid division by zero)
    meaniou = (intersection + eps) / (union + eps)
    dice = (2.0 * intersection + eps) / (total + eps)
    
    # Return as standard Python floats
    return meaniou.item(), dice.item()

In [ ]:
print("Training started...")

# 1. Initialize a variable to track the highest score before the loop begins
best_meaniou = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_meaniou = 0.0
    epoch_dice = 0.0
    
    # Progress bar for the batches
    loop = tqdm(dataloader, desc=f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    
    for images, masks in loop:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        # Forward pass
        predictions = model(images)
        
        # Calculate Loss
        loss = criterion(predictions, masks)
        
        # Calculate Metrics using our new function
        meaniou, dice = calculate_metrics(predictions, masks)
        
        # Backward pass & Optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate totals for the epoch
        epoch_loss += loss.item()
        epoch_meaniou += meaniou
        epoch_dice += dice
        
        # Live update the progress bar with all three metrics
        loop.set_postfix(loss=loss.item(), meaniou=meaniou, dice=dice)
        
    # Calculate averages across the entire epoch
    avg_loss = epoch_loss / len(dataloader)
    avg_meaniou = epoch_meaniou / len(dataloader)
    avg_dice = epoch_dice / len(dataloader)
    
    # Print the final epoch summary
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Summary -> Loss: {avg_loss:.4f} | Mean IoU: {avg_meaniou:.4f} | Dice: {avg_dice:.4f}")
    
    # -----------------------------------------------------------
    # 2. Checkpoint Logic: Save only if Mean IoU improved
    # -----------------------------------------------------------
    if avg_meaniou > best_meaniou:
        print(f"🌟 Mean IoU improved from {best_meaniou:.4f} to {avg_meaniou:.4f}. Saving new best model to {MODEL_SAVE_PATH}...")
        best_meaniou = avg_meaniou
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
    else:
        print(f"Mean IoU did not improve. Current best remains: {best_meaniou:.4f}.")
    print("-" * 50)